In [133]:
import pandas as pd
from pathlib import Path
import os

def select_scenario() -> Path:
    # Get all folder names from the logs folder
    logs_folder = Path(os.path.abspath('')) / 'logs'
    scenarios = [folder for folder in logs_folder.iterdir() if folder.is_dir()]
    print('Select a scenario:')
    for i, folder in enumerate(scenarios):
        print(f"[{i}]: {folder.name}")
    print("Your selection: ")
    selection = int(input())
    return scenarios[selection]

scenario_folder = select_scenario()

Select a scenario:
[0]: soil
Your selection: 


In [134]:
rx_packets_df_col_names = ["Time","Context","Channel freq","Channel no","Rate","Is short preamble","Sender node ID", "Receiver node ID", "Mode","Retries","Ness","Nss","Is short guard interval","Is stbc","Tx power level","Noise","Signal"]
rx_packets_df = pd.read_csv(scenario_folder / 'monitor_sniffer_rx.csv', delimiter=';', names=rx_packets_df_col_names, index_col=False)
rx_packets_df

# Average the signal strength for each time
rx_packets_df_grouped_time = rx_packets_df.groupby('Time')
rx_packets_df_avg_signal = rx_packets_df_grouped_time['Signal'].mean()
rx_packets_df_avg_signal

Time
0.002732      -72.842310
0.002732      -77.494656
0.002732      -80.389300
0.104760      -75.453309
0.207320      -75.453309
                 ...    
1680.590000   -75.453309
1680.800000   -75.453309
1681.000000   -75.453309
1681.210000   -75.453309
1681.410000   -74.097937
Name: Signal, Length: 13940, dtype: float64

In [135]:
#import matplotlib.pyplot as plt

#fig = plt.figure(figsize=(10, 5))
#ax = fig.add_subplot(111)
## Plot a graph where x = time, y = mean signal for that time
#ax.plot(rx_packets_df_avg_signal.index, rx_packets_df_avg_signal.values)
#plt.xlabel('Time (s)')
#lt.ylabel('Average RSSI across all nodes (dBm)')
#plt.show()

In [136]:
node_pos_df = pd.read_csv(scenario_folder / 'course_change.csv', delimiter=';', names=["Time","Context","Type","ID","x","y","z"], index_col=False)
node_pos_df.drop_duplicates(subset=["Time","ID"], keep='last', inplace=True)
node_pos_df.reset_index(drop=True, inplace=True)
node_pos_df

,Time,Context,Type,ID,x,y,z
0,0,/NodeList/0/$ns3::MobilityModel/CourseChange,STA,0,0.4,0.4,-0.3
1,0,/NodeList/1/$ns3::MobilityModel/CourseChange,STA,1,1.2,0.4,-0.3
2,0,/NodeList/2/$ns3::MobilityModel/CourseChange,STA,2,2.0,0.4,-0.3
3,0,/NodeList/3/$ns3::MobilityModel/CourseChange,STA,3,2.8,0.4,-0.3
4,0,/NodeList/4/$ns3::MobilityModel/CourseChange,STA,4,3.6,0.4,-0.3
...,...,...,...,...,...,...,...
196,0,/NodeList/196/$ns3::MobilityModel/CourseChange,STA,196,1.2,10.8,-0.3
197,0,/NodeList/197/$ns3::MobilityModel/CourseChange,STA,197,2.0,10.8,-0.3
198,0,/NodeList/198/$ns3::MobilityModel/CourseChange,STA,198,2.8,10.8,-0.3
199,0,/NodeList/199/$ns3::MobilityModel/CourseChange,STA,199,3.6,10.8,-0.3


In [137]:
avg_signal_per_node = rx_packets_df.groupby('Receiver node ID')['Signal'].mean()
worst_signal = avg_signal_per_node.max()

avg_signal_per_node_offset = avg_signal_per_node - worst_signal
avg_signal_per_node_offset

Receiver node ID
0      -9.774464
1      -9.287787
2      -8.845142
3      -8.487796
4      -8.189070
         ...    
196    -8.634070
197    -7.942373
198    -7.578351
199    -7.230160
200   -19.517953
Name: Signal, Length: 201, dtype: float64

In [138]:
import tkinter
import matplotlib
from matplotlib.backends.backend_tkagg import *
matplotlib.use("TkAgg")

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

root = tkinter.Tk()
root.wm_title("3D signal strength plot")

# Map 'Type' to colors
color_map = {'AP': 'red', 'STA': 'blue'}
colors = node_pos_df['Type'].map(color_map)

fig = plt.figure(dpi=100)

# Create two subplots
ax1 = fig.add_subplot(121, projection='3d')
ax2 = fig.add_subplot(122, projection='3d')

# Plot scatter on first subplot
ax1.scatter(node_pos_df['x'], node_pos_df['y'], node_pos_df['z'], c=colors)
ax1.set_title('Node positions')
ax1.set_xlabel('X (cartesian coord, m)')
ax1.set_ylabel('Y (cartesian coord, m)')
ax1.set_zlabel('Z (cartesian coord, m)')

# Plot bars on second subplot, where height is avg_signal_per_node_offset
for i, row in node_pos_df.iterrows():
    node_id = row['ID']
    delta = avg_signal_per_node_offset[node_id]
    # Ignore the outlier node (AP) when calculating color scale
    sta_values = avg_signal_per_node_offset[node_pos_df[node_pos_df['Type'] == 'STA'].index]
    color = plt.cm.coolwarm((sta_values.max() - delta) / (sta_values.max() - sta_values.min()))
    ax2.bar3d(row['x'], row['y'], 0, .5, .5, delta, color=color)
    
#ax2.bar3d(node_pos_df['x'], node_pos_df['y'], node_pos_df['z'], 0.5, 0.5, avg_signal_per_node_offset, cmap='coolwarm')
ax2.set_title('Average signal strength')
ax2.set_xlabel('X (cartesian coord, m)')
ax2.set_ylabel('Y (cartesian coord, m)')
ax2.set_zlabel(f'Signal loss (dBm), offset by {round(worst_signal, 2)}')

# Sync the view limits
ax1.set_xlim(node_pos_df['x'].min(), node_pos_df['x'].max())
ax1.set_ylim(node_pos_df['y'].min(), node_pos_df['y'].max())
ax2.set_xlim(node_pos_df['x'].min(), node_pos_df['x'].max())
ax2.set_ylim(node_pos_df['y'].min(), node_pos_df['y'].max())

# Create canvas and toolbar
canvas = FigureCanvasTkAgg(fig, master=root)
canvas.draw()
toolbar = NavigationToolbar2Tk(canvas, root)
toolbar.update()
canvas.get_tk_widget().pack(side=tkinter.TOP, fill=tkinter.BOTH, expand=1)

# Function to sync rotation
def on_rotate(event):
    if event.inaxes == ax1:
        ax2.view_init(elev=ax1.elev, azim=ax1.azim)
    elif event.inaxes == ax2:
        ax1.view_init(elev=ax2.elev, azim=ax2.azim)
    fig.canvas.draw_idle()

fig.canvas.mpl_connect('motion_notify_event', on_rotate)

tkinter.mainloop()

KeyboardInterrupt: 